# Taller 1. Segmentación de prensa histórica con DocLayout-YOLO

**Qué produce:** la página con las regiones detectadas, un recorte por región, y las coordenadas en CSV y JSON.

**Para qué:** decidir si las regiones que propone el modelo corresponden a **unidades documentales** (una noticia, un aviso, un título de sección) o si las fragmentan, las fusionan o las omiten.

**Entorno:** Colab con GPU T4 recomendada (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*). Funciona también sin GPU, más lento. El modelo es **genérico para documentos** (artículos científicos, libros, revistas): no fue entrenado con diarios históricos.

## 0. Instalar dependencias (solo Colab)

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "doclayout-yolo==0.0.4", "pypdfium2"], check=True)
    print("Instalado doclayout-yolo 0.0.4")
else:
    print("Fuera de Colab no se instala nada: preparar el entorno según cuadernos/README.md.")

## 1. Elegir las imágenes

Por defecto se procesa la **página común del taller** (*Centro Gallego*, Montevideo, 1917): la misma página en los cuatro cuadernos permite comparar modelos. Cambiá `SELECCION` para trabajar con otras fuentes:

| Valor | Qué procesa |
|---|---|
| `"comun"` | la página común (recomendado para la primera pasada) |
| `"prensa"` | todas las páginas de prensa del taller |
| `"manuscritos"` | cartas, actas y otros manuscritos del taller |
| `"todas"` | prensa y manuscritos |
| `"propias"` | tus imágenes o PDF; en Colab aparecen dos botones para subir **varios archivos** o **una carpeta completa**; fuera de Colab se leen de `mis_imagenes/` (incluidas sus subcarpetas) |

Cada página de un PDF se convierte en una imagen PNG (resolución `DPI_PDF = 200`) y se procesa como una página más.

**No subas material sensible o con restricciones de uso** a servicios externos.

In [ ]:
SELECCION = "comun"

In [ ]:
import csv
import sys
import urllib.request
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

EN_COLAB = "google.colab" in sys.modules
REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/imagenes/"
BASE = Path("/content/taller") if EN_COLAB else Path.cwd()
CARPETA_IMAGENES = BASE / "imagenes"
EXTENSIONES_IMAGEN = {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}
EXTENSIONES = EXTENSIONES_IMAGEN | {".pdf"}
DPI_PDF = 200  # resolución con la que cada página de un PDF se convierte en imagen


def traer(relativa):
    """Ruta local de un archivo del taller; en Colab lo descarga del repositorio la primera vez."""
    destino = CARPETA_IMAGENES / relativa
    if EN_COLAB and not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + relativa, destino)
    if not destino.exists():
        raise FileNotFoundError(f"No encuentro {destino}. Abrí el cuaderno desde la carpeta cuadernos/ del repositorio.")
    return destino


def mostrar(imagen, ancho=1000):
    """Muestra una imagen PIL reducida a un ancho cómodo para la pantalla."""
    if imagen.width > ancho:
        imagen = imagen.resize((ancho, round(imagen.height * ancho / imagen.width)))
    display(imagen)


def admitido(nombre):
    """Imágenes y PDF, sin archivos ocultos (por ejemplo, los `._foto.jpg` que deja macOS en las carpetas)."""
    return not Path(nombre).name.startswith(".") and Path(nombre).suffix.lower() in EXTENSIONES


def pdf_a_imagenes(pdf, carpeta):
    """Convierte cada página de un PDF en una imagen PNG y devuelve sus rutas."""
    import pypdfium2 as pdfium

    carpeta.mkdir(parents=True, exist_ok=True)
    documento = pdfium.PdfDocument(pdf)
    paginas = []
    for indice in range(len(documento)):
        pagina = documento[indice]
        destino = carpeta / f"{pdf.stem}_p{indice + 1:03d}.png"
        pagina.render(scale=DPI_PDF / 72).to_pil().convert("RGB").save(destino)
        pagina.close()
        paginas.append(destino)
    documento.close()
    return paginas


SELECTOR_COLAB = """
<div style="display:flex; gap:2em; align-items:center; flex-wrap:wrap">
  <label><b>Elegir archivos</b> (imágenes y PDF):
    <input type="file" id="__ID__-archivos" multiple accept="image/*,.tif,.tiff,.pdf"></label>
  <span><b>o una carpeta completa</b>:
    <input type="file" id="__ID__-carpeta" webkitdirectory style="display:none"
      onchange="document.getElementById('__ID__-estado').textContent = this.files.length
        ? this.files[0].webkitRelativePath.split('/')[0] + ' (' + this.files.length + ' archivos)' : ''">
    <button type="button" onclick="document.getElementById('__ID__-carpeta').click()">Seleccionar carpeta</button>
    <span id="__ID__-estado">Ninguna carpeta seleccionada</span></span>
</div>
<script>
window.subidaTaller = {
  elegir: (id) => new Promise((resolve) => {
    for (const sufijo of ["archivos", "carpeta"]) {
      document.getElementById(id + "-" + sufijo).addEventListener("change", (evento) => {
        window.subidaTaller.lista = Array.from(evento.target.files);
        resolve(window.subidaTaller.lista.map((f) => f.webkitRelativePath || f.name));
      });
    }
  }),
  trozo: (indice, inicio, fin) => new Promise((resolve, reject) => {
    const lector = new FileReader();
    lector.onload = () => resolve(lector.result.split(",")[1] || "");
    lector.onerror = () => reject(lector.error);
    lector.readAsDataURL(window.subidaTaller.lista[indice].slice(inicio, fin));
  }),
  tamano: (indice) => window.subidaTaller.lista[indice].size,
};
</script>
"""


def subir_en_colab(carpeta, trozo=1024 * 1024):
    """Diálogo de Colab para elegir varios archivos o una carpeta; guarda imágenes y PDF en `carpeta`."""
    import base64

    from google.colab import output

    identificador = "subida-" + uuid.uuid4().hex
    display(HTML(SELECTOR_COLAB.replace("__ID__", identificador)))
    nombres = output.eval_js(f'window.subidaTaller.elegir("{identificador}")')
    for indice, nombre in enumerate(nombres):
        if not admitido(nombre):
            print("Ignorado (no es imagen ni PDF):", nombre)
            continue
        destino = carpeta.joinpath(*Path(nombre).parts)  # una carpeta conserva sus subcarpetas
        destino.parent.mkdir(parents=True, exist_ok=True)
        tamano = output.eval_js(f"window.subidaTaller.tamano({indice})")
        with open(destino, "wb") as archivo:
            for inicio in range(0, tamano, trozo):
                datos = output.eval_js(f"window.subidaTaller.trozo({indice}, {inicio}, {inicio + trozo})")
                archivo.write(base64.b64decode(datos))
        print(f"Subido: {nombre} ({tamano / 1e6:.1f} MB)")


FICHAS = {}
if SELECCION == "propias":
    carpeta_propia = BASE / "mis_imagenes"
    carpeta_propia.mkdir(parents=True, exist_ok=True)
    if EN_COLAB:
        subir_en_colab(carpeta_propia)
    IMAGENES = []
    for archivo in sorted(p for p in carpeta_propia.rglob("*") if p.is_file() and admitido(p.name)):
        if archivo.suffix.lower() == ".pdf":
            paginas = pdf_a_imagenes(archivo, BASE / "paginas_pdf")
            print(f"{archivo.name}: {len(paginas)} página(s) convertidas a imagen a {DPI_PDF} dpi")
            IMAGENES += paginas
        else:
            IMAGENES.append(archivo)
else:
    with open(traer("fuentes.csv"), encoding="utf-8-sig") as archivo:
        fichas = list(csv.DictReader(archivo, delimiter=";"))
    filtros = {
        "comun": lambda f: f["pagina_comun"] == "si",
        "prensa": lambda f: f["tipo"] == "prensa",
        "manuscritos": lambda f: f["tipo"] != "prensa",
        "todas": lambda f: True,
    }
    if SELECCION not in filtros:
        raise ValueError(f"SELECCION debe ser uno de {sorted(filtros) + ['propias']}")
    elegidas = [f for f in fichas if filtros[SELECCION](f)]
    IMAGENES = [traer(f["archivo"]) for f in elegidas]
    FICHAS = {ruta.name: ficha for ruta, ficha in zip(IMAGENES, elegidas)}
    display(pd.DataFrame(elegidas)[["archivo", "tipo", "titulo", "fecha", "lugar", "institucion", "licencia"]])

assert IMAGENES, "No hay imágenes para procesar: revisá SELECCION o la carpeta mis_imagenes/."
print(f"{len(IMAGENES)} imagen(es):", [p.name for p in IMAGENES])

## 2. Cargar el modelo

`CONFIANZA` es el umbral mínimo para aceptar una región: bajarlo muestra más cajas (y más ruido); subirlo deja solo las más seguras. `RESOLUCION` es el tamaño al que el modelo reduce la página antes de analizarla.

In [ ]:
import torch
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download

PESOS = hf_hub_download(repo_id="juliozhao/DocLayout-YOLO-DocStructBench", filename="doclayout_yolo_docstructbench_imgsz1024.pt")
modelo = YOLOv10(PESOS)
DISPOSITIVO = "cuda:0" if torch.cuda.is_available() else "cpu"
CONFIANZA = 0.20
RESOLUCION = 1024

CLASES_ES = {
    "title": "título",
    "plain text": "texto",
    "abandon": "descartable: encabezado, pie, número de página, marginalia",
    "figure": "figura o grabado",
    "figure_caption": "epígrafe de figura",
    "table": "tabla",
    "table_caption": "título de tabla",
    "table_footnote": "nota de tabla",
    "isolate_formula": "fórmula",
    "formula_caption": "número de fórmula",
}
print("Dispositivo:", DISPOSITIVO)
display(pd.DataFrame([{"clase del modelo": n, "significado": CLASES_ES.get(n, "")} for n in modelo.names.values()]))

Las clases provienen del conjunto de entrenamiento (DocStructBench), no de la prensa: no existen «aviso», «noticia» ni «sección». La clase `abandon` agrupa lo que el modelo considera prescindible; para un historiador, la cabecera, la fecha o el número de página **no son prescindibles**.

## 3. Detectar regiones

In [ ]:
import json

from PIL import ImageDraw, ImageFont

SALIDA = BASE / "resultados" / "01_doclayout_yolo"
SALIDA.mkdir(parents=True, exist_ok=True)
COLORES = ["#e63946", "#1d3557", "#2a9d8f", "#e9c46a", "#8338ec", "#f4a261", "#06d6a0", "#ef476f", "#118ab2", "#6c757d"]

filas = []
for pagina in IMAGENES:
    original = Image.open(pagina).convert("RGB")
    resultado = modelo.predict(str(pagina), imgsz=RESOLUCION, conf=CONFIANZA, device=DISPOSITIVO, verbose=False)[0]
    cajas = sorted(
        zip(resultado.boxes.xyxy.cpu().numpy(), resultado.boxes.cls.cpu().numpy(), resultado.boxes.conf.cpu().numpy()),
        key=lambda caja: (caja[0][1], caja[0][0]),
    )
    vista = original.copy()
    lapiz = ImageDraw.Draw(vista)
    grosor = max(3, original.width // 500)
    fuente = ImageFont.load_default(size=max(16, original.width // 90))
    carpeta = SALIDA / pagina.stem / "recortes"
    carpeta.mkdir(parents=True, exist_ok=True)
    for numero, (xyxy, clase, confianza) in enumerate(cajas, 1):
        x1, y1, x2, y2 = (int(v) for v in xyxy)
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(original.width, x2), min(original.height, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        etiqueta = modelo.names[int(clase)]
        color = COLORES[int(clase) % len(COLORES)]
        lapiz.rectangle((x1, y1, x2, y2), outline=color, width=grosor)
        lapiz.text((x1 + grosor, y1 + grosor), f"{numero} {etiqueta}", fill=color, font=fuente, stroke_width=3, stroke_fill="white")
        recorte = carpeta / f"{numero:03d}_{etiqueta.replace(' ', '_')}.png"
        original.crop((x1, y1, x2, y2)).save(recorte)
        filas.append({
            "pagina": pagina.name, "region": numero, "clase": etiqueta, "clase_es": CLASES_ES.get(etiqueta, ""),
            "confianza": round(float(confianza), 3), "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "recorte": recorte.relative_to(SALIDA).as_posix(), "evaluacion_humana": "",
        })
    vista.save(SALIDA / pagina.stem / "regiones.png")
    print(f"{pagina.name}: {len(cajas)} regiones")
    mostrar(vista)

regiones = pd.DataFrame(filas)
regiones.to_csv(SALIDA / "regiones.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
(SALIDA / "regiones.json").write_text(json.dumps(filas, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado en:", SALIDA)

Los números de las cajas siguen la **posición vertical** de cada región (de arriba hacia abajo), no un orden de lectura: el modelo detecta regiones, pero no sabe en qué secuencia se leen las columnas.

## 4. Resumen por clase y recortes

In [ ]:
import matplotlib.pyplot as plt

if regiones.empty:
    print("No se detectaron regiones con CONFIANZA =", CONFIANZA)
else:
    display(regiones.pivot_table(index="pagina", columns="clase", values="region", aggfunc="count", fill_value=0))
    muestra = regiones.head(12)
    figura, ejes = plt.subplots(3, 4, figsize=(16, 12))
    for eje in ejes.flat:
        eje.axis("off")
    for eje, (_, fila) in zip(ejes.flat, muestra.iterrows()):
        eje.imshow(Image.open(SALIDA / fila["recorte"]))
        eje.set_title(f"{fila['region']} · {fila['clase']} · {fila['confianza']:.2f}", fontsize=10)
    plt.tight_layout()
    plt.show()

## 5. Evaluar la segmentación

Abrí `regiones.csv` (se abre directamente en Excel) o la tabla `regiones` y completá la columna `evaluacion_humana` con uno de estos valores: `correcta`, `fragmenta` (parte una unidad en varias cajas), `fusiona` (une unidades distintas), `clase_erronea` u `omite` (agregá una fila para lo que el modelo no vio).

**Preguntas para la discusión**

1. ¿Un recorte equivale a una noticia? ¿Qué pasa con los avisos, los títulos de sección y los grabados?
2. ¿Qué quedó clasificado como `abandon`? ¿Qué información histórica contiene?
3. Con estos recortes, ¿podría reconstruirse el orden de lectura de la página?
4. ¿Qué clases faltan para describir esta publicación? ¿Cómo se comporta el modelo con los manuscritos?

## Registro de mediaciones

Toda transcripción automática depende de la herramienta, la versión, la configuración y el equipo que la produjeron. Esta celda guarda esos datos en `procesamiento.json`, junto a los resultados, para poder citarlos y reproducirlos.

In [ ]:
import datetime
import importlib.metadata as metadata
import json
import platform

registro = {
    "herramienta": "DocLayout-YOLO",
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else platform.platform(),
    "python": platform.python_version(),
    "dispositivo": DISPOSITIVO,
    "imagenes": [p.name for p in IMAGENES],
    "versiones": {p: metadata.version(p) for p in ("doclayout-yolo", "torch")},
    "modelo": "juliozhao/DocLayout-YOLO-DocStructBench (doclayout_yolo_docstructbench_imgsz1024.pt)",
    "parametros": {"confianza": CONFIANZA, "resolucion": RESOLUCION},
}
(SALIDA / "procesamiento.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

## Descargar resultados

In [ ]:
import shutil

if EN_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(f"/content/resultados_01_doclayout_yolo", "zip", SALIDA))
else:
    print("Resultados en:", SALIDA.resolve())